# Assignment 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, balanced_accuracy_score, accuracy_score,
                             pairwise_distances)

SEED = 11


In [ ]:
df = pd.read_csv('networkTraffic.csv', na_values='?')
target = df.columns[-1]

feature_metadata = pd.read_csv('features.csv')
feature_metadata.columns = feature_metadata.columns.str.strip()
metadata_types = dict(zip(feature_metadata['Name'].str.lower(),
                          feature_metadata['Type'].str.strip().str.lower()))
predictor_columns = list(df.columns[:-1])
categorical = [c for c in predictor_columns if metadata_types.get(c.lower()) == 'nominal']
binary = [c for c in predictor_columns if metadata_types.get(c.lower()) == 'binary']

predictor_missing = df[predictor_columns].isna().sum()

print(f'Rows: {len(df):,}; predictors: {df.shape[1]-1}; target: {target}')
print('Nominal predictors:', categorical, '| binary indicators:', binary)
print('Missing predictors:', predictor_missing[predictor_missing.gt(0)].to_dict(),
      '| missing target values:', int(df[target].isna().sum()))
display(df.head())

### Quality issues check 1: Missingness, uniqueness/cardinality, range/scale

In [ ]:
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_n': df.isna().sum(),
    'missing_pct': 100 * df.isna().mean(),
    'unique_n': df.nunique(dropna=True)
})
display(audit.query('missing_n > 0 or unique_n == @df.shape[0]'))

class_map = pd.read_csv('attack_category_map.csv')
name_by_code = dict(zip(class_map['Mapping'], class_map['Attack Category Name']))
class_counts = df[target].value_counts().sort_index().rename(index=name_by_code)
display(pd.DataFrame({'count': class_counts, 'percent': 100*class_counts/len(df)}))
ax = class_counts.plot.bar(figsize=(10,4), title='Original target distribution')
ax.set(ylabel='observations', xlabel='class')
plt.tight_layout()
plt.savefig('data_audit.png')
plt.show()

numeric_all = [c for c in df.select_dtypes(include=np.number).columns if c != target]
ranges = df[numeric_all].agg(['min','median','max']).T
display(ranges.sort_values('max', ascending=False).head(12))

# may be outliers, but a rough check really
q1, q3 = df[numeric_all].quantile(0.25), df[numeric_all].quantile(0.75)
iqr = q3 - q1
potential_outlier = (df[numeric_all].lt(q1 - 1.5*iqr) |
                     df[numeric_all].gt(q3 + 1.5*iqr))
outlier_summary = pd.DataFrame({
    'potential_outlier_n': potential_outlier.sum(),
    'potential_outlier_pct': 100*potential_outlier.mean()
}).sort_values('potential_outlier_pct', ascending=False)
display(outlier_summary.head(20))

numeric_skew = df[numeric_all].skew()
skewed_numeric = [c for c in numeric_all if c not in binary + ['id']
                  and numeric_skew[c] > 1
                  and outlier_summary.loc[c, 'potential_outlier_pct'] > 1]
display(pd.DataFrame({'skewness': numeric_skew[skewed_numeric],
                      'outlier_pct': outlier_summary.loc[skewed_numeric,
                                                          'potential_outlier_pct']})
        .sort_values('skewness', ascending=False))

In [ ]:
class_map

## Quality issue check 2: Correlations between predictors

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    """Bias-corrected Cramer's V for two nominal columns (Bergsma 2013), not mine"""
    x = pd.Series(x)
    y = pd.Series(y)
    mask = x.notna() & y.notna()
    contingency = pd.crosstab(x[mask], y[mask])
    n = mask.sum()
    chi2 = chi2_contingency(contingency, correction=False)[0]
    r, k = contingency.shape
    phi2 = chi2 / n
    phi2_corr = max(0, phi2 - (r - 1) * (k - 1) / (n - 1))
    r_corr = r - (r - 1) ** 2 / (n - 1)
    k_corr = k - (k - 1) ** 2 / (n - 1)
    denom = min(r_corr - 1, k_corr - 1)
    return float(np.sqrt(phi2_corr / denom)) if denom > 0 else 0.0

def check_nominal_association(X_fit, nominal_columns=('proto', 'service', 'state')):
    pairs = []
    for i, left in enumerate(nominal_columns):
        for right in nominal_columns[i+1:]:
            v = cramers_v(X_fit[left], X_fit[right])
            pairs.append({'feature_1': left, 'feature_2': right, 'cramers_v': v})
    return pd.DataFrame(pairs)

check_nominal_association(df)

In [ ]:
missingness_by_proto = df.groupby('proto')['service'].apply(lambda s: s.isna().mean())
missingness_by_proto.sort_values(ascending=False)

In [ ]:
df

In [ ]:
"""
Make upper triangular/non-duplicating pairwise correlation table; exclude diag of course
"""
corr = df[numeric_all].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong_pairs = (upper.stack().rename('abs_r').reset_index()
                .rename(columns={'level_0':'feature_1','level_1':'feature_2'})
                .query('abs_r >= 0.90').sort_values('abs_r', ascending=False))
display(strong_pairs.head(20))
print('Unique id values:', df['id'].nunique(), 'of', len(df))

## 3. Model-specific preprocessing pipelines

In [ ]:
X = df.drop(columns=target)
y = df[target].astype(int)
tree_drop = ['id']
tree_cols = [c for c in X.columns if c not in tree_drop]

def shepard_predictions(distances, neighbour_indices, training_labels, k_values):
    """Predict several k values from one neighbour search using exact 1/d (shephard) voting scheme"""
    classes, encoded_labels = np.unique(np.asarray(training_labels),
                                         return_inverse=True)
    neighbour_classes = encoded_labels[neighbour_indices]
    outputs = {}
    for k in k_values:
        selected_distance = distances[:, :k]
        selected_class = neighbour_classes[:, :k]
        weights = np.divide(1., selected_distance,
                            out=np.zeros_like(selected_distance),
                            where=selected_distance != 0)
        exact = selected_distance == 0
        rows = exact.any(axis=1)
        weights[rows] = exact[rows]
        votes = np.column_stack([
            (weights * (selected_class == class_number)).sum(axis=1)
            for class_number in range(len(classes))
        ])
        outputs[k] = classes[votes.argmax(axis=1)]
    return outputs

class GowerPreprocessor:
    def __init__(self, numerical_log, numerical_plain, binary_columns, nominal_columns):
        self.numerical_log = numerical_log
        self.numerical_plain = numerical_plain
        self.binary_columns = binary_columns
        self.nominal_columns = nominal_columns

    def _numerical_frame(self, X):
        values = X[self.numerical_log + self.numerical_plain].astype(float).copy()
        if self.numerical_log:
            values.loc[:, self.numerical_log] = np.log1p(values[self.numerical_log])
        return values

    def fit(self, X):
        self.scaler_ = MinMaxScaler(clip=True).fit(self._numerical_frame(X))
        self.category_maps_ = {
            column: {value: index for index, value in enumerate(
                pd.unique(X[column].dropna()))}
            for column in self.nominal_columns
        }
        self.service_index_ = self.nominal_columns.index('service')
        self.original_feature_count_ = (len(self.numerical_log) +
            len(self.numerical_plain) + len(self.binary_columns) +
            len(self.nominal_columns))
        return self

    def transform(self, X):
        scaled = self.scaler_.transform(self._numerical_frame(X)).astype(np.float32)
        binary_values = X[self.binary_columns].to_numpy(dtype=np.float32)
        numerical = np.column_stack([scaled, binary_values])
        categorical_codes = np.column_stack([
            X[column].map(self.category_maps_[column]).fillna(-1).to_numpy(dtype=np.int32)
            for column in self.nominal_columns
        ])
        service_observed = X['service'].notna().to_numpy()
        return numerical, categorical_codes, service_observed

    def fit_transform(self, X):
        return self.fit(X).transform(X)

def gower_neighbours(training, queries, k, original_feature_count,
                     service_index, chunk_size=250):
    train_num, train_cat, train_service = training
    query_num, query_cat, query_service = queries
    all_distances, all_indices = [], []
    for start in range(0, len(query_num), chunk_size):
        stop = min(start + chunk_size, len(query_num))
        distance = pairwise_distances(
            query_num[start:stop], train_num, metric='manhattan', n_jobs=-1)
        for column_index in range(train_cat.shape[1]):
            if column_index == service_index:
                continue
            distance += (query_cat[start:stop, column_index, None] !=
                         train_cat[None, :, column_index])
        jointly_observed = (query_service[start:stop, None] &
                            train_service[None, :])
        service_mismatch = (query_cat[start:stop, service_index, None] !=
                            train_cat[None, :, service_index])
        distance += jointly_observed * service_mismatch
        distance /= original_feature_count - (~jointly_observed)
        nearest = np.argpartition(distance, kth=k - 1, axis=1)[:, :k]
        nearest_distance = np.take_along_axis(distance, nearest, axis=1)
        order = np.argsort(nearest_distance, axis=1)
        all_indices.append(np.take_along_axis(nearest, order, axis=1))
        all_distances.append(np.take_along_axis(nearest_distance, order, axis=1))
    return np.vstack(all_distances), np.vstack(all_indices)

tree_nominal_missing = ['service']
tree_nominal_low = ['state']
tree_nominal_high = ['proto']
tree_binary = binary
tree_numeric = [c for c in tree_cols
                if c not in tree_nominal_missing + tree_nominal_low +
                tree_nominal_high + tree_binary]
tree_prep = ColumnTransformer([
    ('num', 'passthrough', tree_numeric),
    ('binary', 'passthrough', tree_binary),
    ('state', OneHotEncoder(handle_unknown='ignore'), tree_nominal_low),
    ('proto', TargetEncoder(target_type='multiclass', smooth='auto', cv=5,
                            shuffle=True, random_state=SEED), tree_nominal_high),
    ('service', Pipeline([
        ('missing_category', SimpleImputer(strategy='constant',
                                           fill_value='__MISSING__')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), tree_nominal_missing)
])
tree_pipe = Pipeline([
    ('prep', tree_prep),
    ('model', DecisionTreeClassifier(random_state=SEED))
])

## Hyperparameter search and evaluation


In [ ]:
SELECTION_SIZE = 20_000
EVALUATION_SIZE = 40_000
selection_index, evaluation_index = train_test_split(
    X.index, train_size=SELECTION_SIZE, test_size=EVALUATION_SIZE,
    stratify=y, random_state=SEED)
X_select, y_select = X.loc[selection_index], y.loc[selection_index]
X_evaluation, y_evaluation = X.loc[evaluation_index], y.loc[evaluation_index]
selection_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
evaluation_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('Hyperparameter selection:', X_select.shape,
      '| CV5 evaluation:', X_evaluation.shape)
display(pd.DataFrame({'full': y.value_counts().sort_index(),
                      'selection': y_select.value_counts().sort_index(),
                      'evaluation': y_evaluation.value_counts().sort_index()})
        .fillna(0).astype(int))

def correlation_ratio(values, labels):
    values = pd.Series(values, dtype=float)
    labels = pd.Series(np.asarray(labels), index=values.index)
    centre = values.mean()
    total_ss = ((values - centre) ** 2).sum()
    if total_ss == 0:
        return 0.0
    grouped = pd.DataFrame({'value': values, 'label': labels}).groupby('label')['value']
    between_ss = sum(len(group) * (group.mean() - centre) ** 2
                     for _, group in grouped)
    return float(np.sqrt(between_ss / total_ss))



def select_knn_features(X_fit, y_fit, threshold=0.995, min_target_eta=None):
    numeric_screen = [c for c in X_fit.select_dtypes(include=np.number).columns
                      if c != 'id']
    absolute_corr = X_fit[numeric_screen].corr().abs()
    adjacent = {c: set() for c in numeric_screen}
    for left_index, left in enumerate(numeric_screen):
        for right in numeric_screen[left_index + 1:]:
            if absolute_corr.loc[left, right] >= threshold:
                adjacent[left].add(right)
                adjacent[right].add(left)
    correlated_groups, visited = [], set()
    for feature in numeric_screen:
        if feature in visited or not adjacent[feature]:
            continue
        stack, component = [feature], []
        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)
            component.append(current)
            stack.extend(adjacent[current] - visited)
        correlated_groups.append(sorted(component))
    eta = {c: correlation_ratio(X_fit[c], y_fit) for c in numeric_screen}
    winners = [max(group, key=lambda c: (eta[c], c))
               for group in correlated_groups]
    correlation_losers = sorted(
        feature for group, winner in zip(correlated_groups, winners)
        for feature in group if feature != winner)

    relevance_losers = sorted(
        feature for feature in numeric_screen
        if min_target_eta is not None
        and feature not in correlation_losers
        and eta[feature] < min_target_eta)
    dropped = ['id'] + sorted(set(correlation_losers + relevance_losers))
    selected = [c for c in X_fit.columns if c not in dropped]
    summary_rows = [
        {'selection_reason': f'|r| >= {threshold:g} near-duplicate group',
         'features_considered': ', '.join(group),
         'decision': f'provisionally retain {winner}',
         'target_eta': eta[winner]}
        for group, winner in zip(correlated_groups, winners)
    ]
    summary_rows.extend(
        {'selection_reason': f'target eta < {min_target_eta:g}',
         'features_considered': feature,
         'decision': 'remove',
         'target_eta': eta[feature]}
        for feature in relevance_losers)
    summary = pd.DataFrame(summary_rows)
    return selected, dropped, summary

def select_knn_logs(X_fit, selected):
    numerical = [c for c in selected if c not in categorical + binary]
    q1, q3 = X_fit[numerical].quantile(.25), X_fit[numerical].quantile(.75)
    iqr = q3 - q1
    outlier_pct = 100 * ((X_fit[numerical].lt(q1 - 1.5 * iqr)) |
                         (X_fit[numerical].gt(q3 + 1.5 * iqr))).mean()
    skew = X_fit[numerical].skew()
    return [c for c in numerical if skew[c] > 0.5 and outlier_pct[c] > 1]

def build_knn_preprocessor(X_fit, y_fit):
    selected, dropped, summary = select_knn_features(X_fit, y_fit)
    nominal = [c for c in categorical if c in selected]
    binary_selected = [c for c in binary if c in selected]
    numerical = [c for c in selected if c not in nominal + binary_selected]
    numerical_log = select_knn_logs(X_fit, selected)
    numerical_plain = [c for c in numerical if c not in numerical_log]
    prep = GowerPreprocessor(
        numerical_log=numerical_log, numerical_plain=numerical_plain,
        binary_columns=binary_selected, nominal_columns=nominal)
    return prep, selected, dropped, summary, numerical_log

_, knn_cols, knn_drop, selection_summary, knn_numeric_log = build_knn_preprocessor(X_select, y_select)
display(selection_summary)
print('Dropped from final kNN:', knn_drop)
print('Log-transformed in final kNN:', knn_numeric_log)

### kNN: CV3 model selection and CV5 evaluation

In [ ]:
selection_folds = list(selection_cv.split(X_select, y_select))
knn_candidates = [1, 2, 3, 5, 7, 9, 11, 21, 31]
knn_selection_scores = {k: {'macro_f1': [],
                            'balanced_accuracy': [],
                            'accuracy': []}
                        for k in knn_candidates}
for train_index, holdout_index in selection_folds:
    X_train, y_train = X_select.iloc[train_index], y_select.iloc[train_index]
    X_holdout, y_holdout = (X_select.iloc[holdout_index], y_select.iloc[holdout_index])
    fold_prep, fold_cols, _, _, _ = build_knn_preprocessor(X_train, y_train)
    transformed_train = fold_prep.fit_transform(X_train[fold_cols])
    transformed_holdout = fold_prep.transform(X_holdout[fold_cols])
    distances, indices = gower_neighbours(transformed_train, transformed_holdout, max(knn_candidates), fold_prep.original_feature_count_, fold_prep.service_index_)
    candidate_predictions = shepard_predictions(distances, indices, y_train, knn_candidates)
    for k, prediction in candidate_predictions.items():
        knn_selection_scores[k]['macro_f1'].append(f1_score(y_holdout, prediction, average='macro'))
        knn_selection_scores[k]['balanced_accuracy'].append(balanced_accuracy_score(y_holdout, prediction))
        knn_selection_scores[k]['accuracy'].append(accuracy_score(y_holdout, prediction))
knn_selection_results = pd.DataFrame([
    {'k': k,
     'cv3_holdout_macro_f1_mean': np.mean(scores['macro_f1']),
     'cv3_holdout_macro_f1_sd': np.std(scores['macro_f1'], ddof=1),
     'cv3_holdout_balanced_accuracy_mean': np.mean(scores['balanced_accuracy']),
     'cv3_holdout_accuracy_mean': np.mean(scores['accuracy'])}
    for k, scores in knn_selection_scores.items()
])
best_k = int(knn_selection_results.loc[knn_selection_results['cv3_holdout_macro_f1_mean'].idxmax(), 'k'])
knn_selection_results['selected'] = (knn_selection_results['k'] == best_k)
display(knn_selection_results.sort_values('cv3_holdout_macro_f1_mean', ascending=False).round(4))
print(f'Selected k={best_k}')

knn_evaluation_rows = []
knn_oof_prediction = np.empty(len(y_evaluation), dtype=y_evaluation.dtype)
for fold, (train_index, holdout_index) in enumerate(evaluation_cv.split(X_evaluation, y_evaluation), start=1):
    X_train, y_train = (X_evaluation.iloc[train_index],y_evaluation.iloc[train_index])
    X_holdout, y_holdout = (X_evaluation.iloc[holdout_index],y_evaluation.iloc[holdout_index])
    fold_prep, fold_cols, _, _, _ = build_knn_preprocessor(X_train, y_train)
    transformed_train = fold_prep.fit_transform(X_train[fold_cols])
    transformed_holdout = fold_prep.transform(X_holdout[fold_cols])
    
    train_distances, train_indices = gower_neighbours(transformed_train, transformed_train, best_k, fold_prep.original_feature_count_, fold_prep.service_index_)
    holdout_distances, holdout_indices = gower_neighbours(transformed_train, transformed_holdout, best_k, fold_prep.original_feature_count_, fold_prep.service_index_)
    train_prediction = shepard_predictions(train_distances, train_indices, y_train, [best_k])[best_k]
    holdout_prediction = shepard_predictions(holdout_distances, holdout_indices, y_train, [best_k])[best_k]
    knn_oof_prediction[holdout_index] = holdout_prediction

    for partition, actual, prediction in [('train', y_train, train_prediction), ('holdout', y_holdout, holdout_prediction)]:
        knn_evaluation_rows.append({'fold': fold, 'partition': partition,
                                    'macro_f1': f1_score(actual, prediction, average='macro'),
                                    'balanced_accuracy': balanced_accuracy_score(actual, prediction),
                                    'accuracy': accuracy_score(actual, prediction)})
        
knn_evaluation_folds = pd.DataFrame(knn_evaluation_rows)
knn_evaluation_summary = knn_evaluation_folds.groupby('partition')[['macro_f1', 'balanced_accuracy', 'accuracy']].agg(['mean', 'std'])
print('kNN: fixed-hyperparameter CV5 fold results')
display(knn_evaluation_folds.round(4))
print('kNN: mean training (four folds) and holdout (one fold) performance')
display(knn_evaluation_summary.round(4))

### Decision tree: CV3 model selection and CV5 evaluation

In [ ]:

tree_grid = {
    "model__class_weight": [None, "balanced"],
    "model__criterion": ["entropy", "gini"],
    "model__max_depth": [10, 20, 30, 80, None],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__min_samples_split": [2, 4, 10, 20],
    #"model__ccp_alpha": [0.0, 1e-4, 1e-3, 1e-2, 1e-1] did not help; redundent considering min__samples_""
}
tree_scoring = {'macro_f1': 'f1_macro', 'balanced_accuracy': 'balanced_accuracy', 'accuracy': 'accuracy'}
tree_search = RandomizedSearchCV(tree_pipe, tree_grid, scoring=tree_scoring, refit='macro_f1', cv=selection_cv, n_iter=140, random_state=SEED, n_jobs=-1, return_train_score=True)
tree_search.fit(X_select[tree_cols], y_select)
tree_selection_results = pd.DataFrame(tree_search.cv_results_).sort_values(
    'rank_test_macro_f1')
tree_selection_columns = ['rank_test_macro_f1', 'params', 'mean_test_macro_f1', 'std_test_macro_f1', 'mean_test_balanced_accuracy', 'mean_test_accuracy']
display(tree_selection_results[tree_selection_columns].head(10).round(4))
selected_tree_params = tree_search.best_params_
print('Selected tree hyperparameters:', selected_tree_params)
print(f'CV3 mean holdout macro-F1: ' f'({tree_search.best_score_:.4f})')

tree_evaluation_rows = []
tree_oof_prediction = np.empty(len(y_evaluation), dtype=y_evaluation.dtype)
for fold, (train_index, holdout_index) in enumerate(evaluation_cv.split(X_evaluation, y_evaluation), start=1):
    X_train, y_train = (X_evaluation.iloc[train_index], y_evaluation.iloc[train_index])
    X_holdout, y_holdout = (X_evaluation.iloc[holdout_index], y_evaluation.iloc[holdout_index])

    fold_prep = clone(tree_search.best_estimator_.named_steps['prep'])
    transformed_train = fold_prep.fit_transform(X_train[tree_cols], y_train)
    transformed_holdout = fold_prep.transform(X_holdout[tree_cols])

    fold_model = clone(tree_search.best_estimator_.named_steps['model'])
    fold_model.fit(transformed_train, y_train)

    train_prediction = fold_model.predict(transformed_train)
    holdout_prediction = fold_model.predict(transformed_holdout)
    tree_oof_prediction[holdout_index] = holdout_prediction

    for partition, actual, prediction in [('train', y_train, train_prediction), ('holdout', y_holdout, holdout_prediction)]:
        tree_evaluation_rows.append({'fold': fold, 'partition': partition,
            'macro_f1': f1_score(actual, prediction, average='macro'),
            'balanced_accuracy': balanced_accuracy_score(actual, prediction),
            'accuracy': accuracy_score(actual, prediction)})
        
tree_evaluation_folds = pd.DataFrame(tree_evaluation_rows)
tree_evaluation_summary = tree_evaluation_folds.groupby('partition')[['macro_f1', 'balanced_accuracy', 'accuracy']].agg(['mean', 'std'])

print('Decision tree: fixed-hyperparameter CV5 fold results')
display(tree_evaluation_folds.round(4))
print('Decision tree: mean training (four folds) and holdout (one fold) performance')
display(tree_evaluation_summary.round(4))


In [ ]:
evaluation_summaries = {'kNN': knn_evaluation_folds, 'Decision tree': tree_evaluation_folds}
mean_performance = pd.concat([
    scores.groupby('partition')[['macro_f1', 'balanced_accuracy', 'accuracy']].mean().assign(model=name).reset_index()
    for name, scores in evaluation_summaries.items()
], ignore_index=True).set_index(['model', 'partition'])

display(mean_performance.round(4))

In [ ]:
labels = sorted(np.unique(y))
oof_predictions = {'kNN': knn_oof_prediction, 'Decision tree': tree_oof_prediction}

for name, prediction in oof_predictions.items():
    print('\n', name, 'pooled CV5 holdout classification report')
    report = classification_report(y_evaluation, prediction, labels=labels, target_names=[name_by_code[i] for i in labels], zero_division=0, output_dict=True)
    display(pd.DataFrame(report).T.round(4))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (name, prediction) in zip(axes, oof_predictions.items()):
    cm = confusion_matrix(y_evaluation, prediction, labels=labels, normalize='true')
    sns.heatmap(cm, cmap='Blues', vmin=0, vmax=1, ax=ax, xticklabels=[name_by_code[i] for i in labels], yticklabels=[name_by_code[i] for i in labels])
    ax.set(title=f'{name}: pooled CV5 holdout confusion matrix', xlabel='predicted', ylabel='actual')
    ax.tick_params(axis='x', rotation=60)
    
plt.tight_layout()
plt.show()